# Mesh Curvature

In [ ]:
import trimesh

import numpy as np
import torch as th
import os
from superfit.optim.curvature import get_points_and_weights
from geolipi.torch_compute import Sketcher, recursive_evaluate
from superfit.utils.config import AlgorithmConfig as AlgConf
from superfit.utils.mesh_preprocess import process_mesh_to_sdf


In [ ]:
# Add mesh path here. 
target_mesh_path = ""
if not os.path.exists(target_mesh_path):
    raise ValueError(f"Mesh path {target_mesh_path} does not exist.")
sketcher = Sketcher(device="cuda", n_dims=3, resolution=256)

In [ ]:
from superfit.utils.mesh_preprocess import cd_based_process_mesh_to_sdf
target_mesh, target_sdf, _ = cd_based_process_mesh_to_sdf(target_mesh_path, sketcher)

In [ ]:


surface_sampled_points, curvature_weights, C_multi = get_points_and_weights(target_mesh, sketcher, n_points=2 * AlgConf.N_SURFACE_POINTS)
curvature_weights = AlgConf.CURVATURE_WEIGHTS_SCALE * curvature_weights


In [ ]:
curvature_weights.min(), curvature_weights.max()

In [ ]:
import cubvh
from superfit.optim.curvature import get_curvature_weights_for_points
BVH = cubvh.cuBVH(target_mesh.vertices, target_mesh.faces)

# Pre-allocate base_coords once
base_coords = sketcher.get_base_coords()
base_curvature_weights = get_curvature_weights_for_points(base_coords, BVH, target_mesh, C_vertex=C_multi)
base_curvature_weights = AlgConf.CURVATURE_WEIGHTS_SCALE * base_curvature_weights
# base_coords = base_coords.unsqueeze(0)#.expand(1, base_coords.shape[0], base_coords.shape[1])



In [ ]:
# If internal points are needed. 
# curvature_weights = th.cat([base_curvature_weights, curvature_weights], dim=0)
# surface_sampled_points = th.cat([base_coords, surface_sampled_points], dim=0)

In [ ]:
# Convert to numpy if tensors (get_points_and_weights returns torch tensors)
mask = curvature_weights > 0.2
surface_sampled_points = surface_sampled_points[mask]
curvature_weights = curvature_weights[mask]
pts = surface_sampled_points.cpu().numpy() if hasattr(surface_sampled_points, "cpu") else np.asarray(surface_sampled_points)
w = curvature_weights.cpu().numpy() if hasattr(curvature_weights, "cpu") else np.asarray(curvature_weights)
w = np.asarray(w).flatten()

# Normalize weights to [0, 1] for coloring (red = high, blue = low)
w_min, w_max = w.min(), w.max()
w_norm = (w - w_min) / (w_max - w_min + 1e-9)

# Blue (low) -> Red (high): RGB linear interpolation
# t=0 -> blue (0,0,255), t=1 -> red (255,0,0)
colors = np.zeros((len(pts), 4), dtype=np.uint8)
colors[:, 0] = (w_norm * 255).astype(np.uint8)   # R
colors[:, 2] = ((1 - w_norm) * 255).astype(np.uint8)  # B
colors[:, 3] = 255  # A

# Build trimesh scene: mesh (semi-transparent) + point cloud
scene = trimesh.Scene()
# Mesh with subtle gray so points stand out
mesh_vis = target_mesh.copy()
mesh_vis.visual.face_colors = [180, 180, 180, 200]
# scene.add_geometry(mesh_vis, geom_name="mesh")

point_cloud = trimesh.points.PointCloud(pts, colors=colors)
scene.add_geometry(point_cloud, geom_name="curvature_points")

# For Jupyter: show in notebook (requires pyglet) or save image
# scene.show()  # opens window
# Or render to image:
# img = scene.save_image(resolution=[1920, 1080]); from IPython.display import Image; display(Image(img))
scene.show()